# 01 清洗：预聚合 CSV → 回归数据

这个 notebook 直接使用当前已经齐全的数据文件，从上到下一步一步清洗，不做缺失文件判断，不封装函数。

输入数据：

- `firm_buy.csv`：购方企业 × 产品
- `firm_sell.csv`：销方企业 × 产品
- `city_buy.csv`：购方地区 × 产品
- `city_sell.csv`：销方地区 × 产品
- `firm_city.csv`：企业 ID × 地区
- `bianma.dta`：合法 9 位产品码
- `full_data.dta`：企业特征

输出数据：

- `invoice_panel.dta`
- `market_conds.dta`
- `firm_chars.dta`


## Step 0 路径和基础设置

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 220)

BASE = Path(r'G:\Kuangyu_Temp\Outsource\productivity')
ROOT = BASE.parent

firm_buy_path = BASE / 'firm_buy.csv'
firm_sell_path = BASE / 'firm_sell.csv'
city_buy_path = BASE / 'city_buy.csv'
city_sell_path = BASE / 'city_sell.csv'
firm_city_path = BASE / 'firm_city.csv'
bianma_path = ROOT / 'bianma.dta'
full_data_path = ROOT / 'full_data.dta'

print('BASE =', BASE)
print('ROOT =', ROOT)

## Step 1 读入所有文件

CSV 里的 ID 和项目代码都按字符串读入。

In [ ]:
na_values = ['', 'NULL', '(Null)', 'null', 'None', 'nan', 'NaN']

firm_buy = pd.read_csv(firm_buy_path, dtype=str, na_values=na_values, encoding='utf-8-sig')
firm_buy.columns = firm_buy.columns.str.strip()
print('firm_buy:', firm_buy.shape, list(firm_buy.columns))
display(firm_buy.head())

firm_sell = pd.read_csv(firm_sell_path, dtype=str, na_values=na_values, encoding='utf-8-sig')
firm_sell.columns = firm_sell.columns.str.strip()
print('firm_sell:', firm_sell.shape, list(firm_sell.columns))
display(firm_sell.head())

city_buy = pd.read_csv(city_buy_path, dtype=str, na_values=na_values, encoding='utf-8-sig')
city_buy.columns = city_buy.columns.str.strip()
print('city_buy:', city_buy.shape, list(city_buy.columns))
display(city_buy.head())

city_sell = pd.read_csv(city_sell_path, dtype=str, na_values=na_values, encoding='utf-8-sig')
city_sell.columns = city_sell.columns.str.strip()
print('city_sell:', city_sell.shape, list(city_sell.columns))
display(city_sell.head())

firm_city = pd.read_csv(firm_city_path, dtype=str, na_values=na_values, encoding='utf-8-sig')
firm_city.columns = firm_city.columns.str.strip()
print('firm_city:', firm_city.shape, list(firm_city.columns))
display(firm_city.head())

## Step 2 统一列名和变量类型

In [ ]:
fb = firm_buy[['购方企业ID', '项目代码', '金额合计', '数量合计']].copy()
fb = fb.rename(columns={'购方企业ID': 'firm_id', '项目代码': 'product_code', '金额合计': 'value', '数量合计': 'qty'})
fb['firm_id'] = fb['firm_id'].astype('string').str.strip()
fb['product_code'] = fb['product_code'].astype('string').str.strip()
fb['value'] = pd.to_numeric(fb['value'], errors='coerce')
fb['qty'] = pd.to_numeric(fb['qty'], errors='coerce')
fb['year'] = 2017

fs = firm_sell[['销方企业ID', '项目代码', '金额合计', '数量合计']].copy()
fs = fs.rename(columns={'销方企业ID': 'firm_id', '项目代码': 'product_code', '金额合计': 'value', '数量合计': 'qty'})
fs['firm_id'] = fs['firm_id'].astype('string').str.strip()
fs['product_code'] = fs['product_code'].astype('string').str.strip()
fs['value'] = pd.to_numeric(fs['value'], errors='coerce')
fs['qty'] = pd.to_numeric(fs['qty'], errors='coerce')
fs['year'] = 2017

cb = city_buy[['购方地区', '项目代码', '金额合计', '数量合计']].copy()
cb = cb.rename(columns={'购方地区': 'city', '项目代码': 'product_code', '金额合计': 'value', '数量合计': 'qty'})
cb['city'] = cb['city'].astype('string').str.strip()
cb['product_code'] = cb['product_code'].astype('string').str.strip()
cb['value'] = pd.to_numeric(cb['value'], errors='coerce')
cb['qty'] = pd.to_numeric(cb['qty'], errors='coerce')
cb['year'] = 2017

cs = city_sell[['销方地区', '项目代码', '金额合计', '数量合计']].copy()
cs = cs.rename(columns={'销方地区': 'city', '项目代码': 'product_code', '金额合计': 'value', '数量合计': 'qty'})
cs['city'] = cs['city'].astype('string').str.strip()
cs['product_code'] = cs['product_code'].astype('string').str.strip()
cs['value'] = pd.to_numeric(cs['value'], errors='coerce')
cs['qty'] = pd.to_numeric(cs['qty'], errors='coerce')
cs['year'] = 2017

fcity = firm_city[['企业ID', '地区']].copy()
fcity = fcity.rename(columns={'企业ID': 'firm_id', '地区': 'city'})
fcity['firm_id'] = fcity['firm_id'].astype('string').str.strip()
fcity['city'] = fcity['city'].astype('string').str.strip()
fcity = fcity.dropna(subset=['firm_id', 'city']).drop_duplicates('firm_id')

print('fb:', fb.shape)
print('fs:', fs.shape)
print('cb:', cb.shape)
print('cs:', cs.shape)
print('fcity:', fcity.shape)

## Step 3 清理项目代码和非正金额 / 数量

规则：项目代码必须是纯数字；右补零到 19 位；取前 9 位作为 `product_id`；删除金额或数量非正的行。

In [ ]:
print('firm_buy before:', len(fb))
fb = fb.dropna(subset=['firm_id', 'product_code', 'value', 'qty']).copy()
fb = fb[fb['product_code'].str.fullmatch(r'\d+', na=False)].copy()
fb['code19'] = fb['product_code'].str.ljust(19, '0').str[:19]
fb['product_id'] = fb['code19'].str[:9]
fb = fb[(fb['value'] > 0) & (fb['qty'] > 0)].copy()
print('firm_buy after:', len(fb), 'unique products:', fb['product_id'].nunique())

print('firm_sell before:', len(fs))
fs = fs.dropna(subset=['firm_id', 'product_code', 'value', 'qty']).copy()
fs = fs[fs['product_code'].str.fullmatch(r'\d+', na=False)].copy()
fs['code19'] = fs['product_code'].str.ljust(19, '0').str[:19]
fs['product_id'] = fs['code19'].str[:9]
fs = fs[(fs['value'] > 0) & (fs['qty'] > 0)].copy()
print('firm_sell after:', len(fs), 'unique products:', fs['product_id'].nunique())

print('city_buy before:', len(cb))
cb = cb.dropna(subset=['city', 'product_code', 'value', 'qty']).copy()
cb = cb[cb['product_code'].str.fullmatch(r'\d+', na=False)].copy()
cb['code19'] = cb['product_code'].str.ljust(19, '0').str[:19]
cb['product_id'] = cb['code19'].str[:9]
cb = cb[(cb['value'] > 0) & (cb['qty'] > 0)].copy()
print('city_buy after:', len(cb), 'unique products:', cb['product_id'].nunique())

print('city_sell before:', len(cs))
cs = cs.dropna(subset=['city', 'product_code', 'value', 'qty']).copy()
cs = cs[cs['product_code'].str.fullmatch(r'\d+', na=False)].copy()
cs['code19'] = cs['product_code'].str.ljust(19, '0').str[:19]
cs['product_id'] = cs['code19'].str[:9]
cs = cs[(cs['value'] > 0) & (cs['qty'] > 0)].copy()
print('city_sell after:', len(cs), 'unique products:', cs['product_id'].nunique())

display(fb.head())

## Step 4 匹配 `bianma.dta`

只保留合法的 9 位产品码。

In [ ]:
bianma = pd.read_stata(bianma_path)
print('bianma:', bianma.shape, list(bianma.columns))

valid_products = set(bianma['product_id'].astype(str).str.strip())
print('valid products:', len(valid_products))

n = len(fb)
fb = fb[fb['product_id'].isin(valid_products)].copy()
print('firm_buy matched:', len(fb), '/', n, f'({len(fb)/n*100:.2f}%)')

n = len(fs)
fs = fs[fs['product_id'].isin(valid_products)].copy()
print('firm_sell matched:', len(fs), '/', n, f'({len(fs)/n*100:.2f}%)')

n = len(cb)
cb = cb[cb['product_id'].isin(valid_products)].copy()
print('city_buy matched:', len(cb), '/', n, f'({len(cb)/n*100:.2f}%)')

n = len(cs)
cs = cs[cs['product_id'].isin(valid_products)].copy()
print('city_sell matched:', len(cs), '/', n, f'({len(cs)/n*100:.2f}%)')

## Step 5 构造企业采购价格

`firm_buy.csv` 已经是企业 × 产品的年度聚合。这里再次按企业 × 产品聚合一次，避免重复行。价格为金额除以数量。

In [ ]:
dv_base = fb.groupby(['firm_id', 'product_id', 'year'], as_index=False).agg(
    value=('value', 'sum'),
    qty=('qty', 'sum'),
    n_rows=('value', 'size')
)
dv_base = dv_base[(dv_base['value'] > 0) & (dv_base['qty'] > 0)].copy()
dv_base['p_buy'] = dv_base['value'] / dv_base['qty']
dv_base['ln_p_buy'] = np.log(dv_base['p_buy'])

# 兼容后面的 Stata 回归脚本。
dv_base['p_net'] = dv_base['p_buy']
dv_base['ln_p_net'] = dv_base['ln_p_buy']
dv_base['ln_qty'] = np.log(dv_base['qty'])

print('dv_base rows:', len(dv_base))
print('unique firms:', dv_base['firm_id'].nunique())
print('unique products:', dv_base['product_id'].nunique())
display(dv_base[['p_buy', 'qty', 'value']].describe(percentiles=[.01, .05, .5, .95, .99]).round(4))
display(dv_base.head())

## Step 6 合并企业地区

用 `firm_city.csv` 把企业采购价格面板补上地区。

In [ ]:
dv = dv_base.merge(fcity, on='firm_id', how='left', indicator=True)
print(dv['_merge'].value_counts(dropna=False))
print('missing city rate:', (dv['_merge'] != 'both').mean())
dv = dv[dv['_merge'] == 'both'].drop(columns='_merge').copy()

print('dv after city merge:', len(dv))
print('unique cities:', dv['city'].nunique())
display(dv.head())

## Step 7 构造市场条件

市场条件在产品 × 地区 × 年份层面。

- `city_buy.csv` 提供地区采购总金额和总数量。
- `city_sell.csv` 提供地区销售总金额和总数量。
- `firm_buy.csv` 合并 `firm_city.csv` 后计算 `n_buyers`。
- `firm_sell.csv` 合并 `firm_city.csv` 后计算 `n_sellers`。

In [ ]:
mkt_buy = cb.groupby(['product_id', 'city', 'year'], as_index=False).agg(
    mkt_value=('value', 'sum'),
    mkt_qty=('qty', 'sum'),
    n_rows_buy=('value', 'size')
)
mkt_buy = mkt_buy[(mkt_buy['mkt_value'] > 0) & (mkt_buy['mkt_qty'] > 0)].copy()
mkt_buy['p_mkt'] = mkt_buy['mkt_value'] / mkt_buy['mkt_qty']
mkt_buy['ln_p_mkt'] = np.log(mkt_buy['p_mkt'])
mkt_buy['ln_mkt_qty'] = np.log(mkt_buy['mkt_qty'])

mkt_sell = cs.groupby(['product_id', 'city', 'year'], as_index=False).agg(
    sell_value=('value', 'sum'),
    sell_qty=('qty', 'sum'),
    n_rows_sell=('value', 'size')
)
mkt_sell = mkt_sell[(mkt_sell['sell_value'] > 0) & (mkt_sell['sell_qty'] > 0)].copy()
mkt_sell['p_sell_mkt'] = mkt_sell['sell_value'] / mkt_sell['sell_qty']

buyers = fb[['firm_id', 'product_id', 'year']].merge(fcity, on='firm_id', how='left')
buyers = buyers.dropna(subset=['city'])
buyers = buyers.groupby(['product_id', 'city', 'year'], as_index=False).agg(n_buyers=('firm_id', 'nunique'))
buyers['ln_n_buyers'] = np.log(buyers['n_buyers'])

sellers = fs[['firm_id', 'product_id', 'year']].merge(fcity, on='firm_id', how='left')
sellers = sellers.dropna(subset=['city'])
sellers = sellers.groupby(['product_id', 'city', 'year'], as_index=False).agg(n_sellers=('firm_id', 'nunique'))
sellers['ln_n_sellers'] = np.log(sellers['n_sellers'])

mkt = mkt_buy.merge(buyers, on=['product_id', 'city', 'year'], how='left')
mkt = mkt.merge(mkt_sell, on=['product_id', 'city', 'year'], how='left')
mkt = mkt.merge(sellers, on=['product_id', 'city', 'year'], how='left')

print('mkt rows:', len(mkt))
print('unique products:', mkt['product_id'].nunique())
print('unique cities:', mkt['city'].nunique())
print('missing n_buyers:', mkt['n_buyers'].isna().mean())
print('missing n_sellers:', mkt['n_sellers'].isna().mean())
display(mkt[['mkt_qty', 'p_mkt', 'n_buyers', 'sell_qty', 'p_sell_mkt', 'n_sellers']].describe(percentiles=[.01, .05, .5, .95, .99]).round(4))
display(mkt.head())

## Step 8 读取企业特征

从 `full_data.dta` 提取 firm × year 层面的企业规模、产品数和中介企业标记。

In [ ]:
sample_full = next(pd.read_stata(full_data_path, chunksize=1))
full_cols = list(sample_full.columns)
print('full_data columns:', len(full_cols))
print(full_cols[:80])

usecols = ['firm_id', 'year', 'firm_total_output', 'firm_total_outsource', 'n_products', 'is_intermediary']

parts = []
for ch in pd.read_stata(full_data_path, columns=usecols, chunksize=500_000):
    ch = ch.drop_duplicates(['firm_id', 'year'])
    parts.append(ch)

firm_chars = pd.concat(parts, ignore_index=True).drop_duplicates(['firm_id', 'year'])
firm_chars['firm_id'] = firm_chars['firm_id'].astype(str).str.strip()
firm_chars['year'] = firm_chars['year'].astype(int)
firm_chars['firm_total_output'] = pd.to_numeric(firm_chars['firm_total_output'], errors='coerce')
firm_chars['firm_total_outsource'] = pd.to_numeric(firm_chars['firm_total_outsource'], errors='coerce')
firm_chars['ln_firm_output'] = np.log(firm_chars['firm_total_output'].clip(lower=1))
firm_chars['ln_firm_outsource'] = np.log(firm_chars['firm_total_outsource'].clip(lower=1))

print('firm_chars rows:', len(firm_chars))
print('unique firms:', firm_chars['firm_id'].nunique())
display(firm_chars.head())

## Step 9 合并覆盖率诊断

In [ ]:
check_mkt = dv.merge(
    mkt[['product_id', 'city', 'year', 'ln_p_mkt', 'ln_mkt_qty', 'ln_n_buyers', 'ln_n_sellers']],
    on=['product_id', 'city', 'year'],
    how='left',
    indicator=True
)
print('dv × market_conds')
print(check_mkt['_merge'].value_counts(dropna=False))
print('market missing rate:', (check_mkt['_merge'] != 'both').mean())

check_firm = dv.merge(
    firm_chars[['firm_id', 'year', 'ln_firm_output', 'n_products', 'is_intermediary']],
    on=['firm_id', 'year'],
    how='left',
    indicator=True
)
print('dv × firm_chars')
print(check_firm['_merge'].value_counts(dropna=False))
print('firm_chars missing rate:', (check_firm['_merge'] != 'both').mean())

## Step 10 导出 Stata 文件

In [ ]:
invoice_panel = dv[['firm_id', 'product_id', 'city', 'year', 'value', 'qty', 'p_buy', 'ln_p_buy', 'p_net', 'ln_p_net', 'ln_qty', 'n_rows']].copy()
invoice_panel.to_stata(BASE / 'invoice_panel.dta', write_index=False, version=118)
print('saved invoice_panel.dta:', len(invoice_panel))

market_cols = [
    'product_id', 'city', 'year',
    'mkt_value', 'mkt_qty', 'p_mkt', 'ln_p_mkt', 'ln_mkt_qty',
    'n_buyers', 'ln_n_buyers',
    'sell_value', 'sell_qty', 'p_sell_mkt',
    'n_sellers', 'ln_n_sellers',
    'n_rows_buy', 'n_rows_sell'
]
market_conds = mkt[market_cols].copy()
market_conds.to_stata(BASE / 'market_conds.dta', write_index=False, version=118)
print('saved market_conds.dta:', len(market_conds))

firm_cols = [
    'firm_id', 'year', 'firm_total_output', 'firm_total_outsource',
    'n_products', 'is_intermediary', 'ln_firm_output', 'ln_firm_outsource'
]
firm_out = firm_chars[firm_cols].copy()
firm_out.to_stata(BASE / 'firm_chars.dta', write_index=False, version=118)
print('saved firm_chars.dta:', len(firm_out))

print('done')